#SVM Inference Notebook

This notebook demonstrates how to use a pre-trained Support Vector Machine (SVM) model for relation classification on new text samples. It loads the trained SVM and XGBoost models, processes input sentences by extracting POS tags and dependency features, converts them into TF-IDF representations, and ensures feature compatibility before making predictions.

By the end of this notebook, you'll be able to input a sentence containing two entities and receive predicted relation types from both SVM and XGBoost models.

# Installing all the required resources

In [5]:
!pip install nltk spacy joblib xgboost scikit-learn  # Installs necessary dependencies for NLP and ML tasks

# Import required libraries
import nltk  # Natural Language Toolkit for text processing
import joblib  # Library for saving and loading trained models
import spacy  # SpaCy for NLP tasks
import numpy as np  # NumPy for numerical operations and array manipulations
from nltk import word_tokenize, pos_tag  # NLTK functions for tokenizing text and extracting POS tags
from scipy.sparse import hstack, csr_matrix  # Sparse matrix operations for efficient storage and processing of features
import gensim.downloader as api # Import the Gensim downloader to load pre-trained word embeddings (e.g., Word2Vec, GloVe, FastText)

# Download necessary NLTK resources
nltk.download("punkt")  # Sentence tokenizer for splitting text into words/sentences
nltk.download("punkt_tab")  # Tab tokenizer
nltk.download("averaged_perceptron_tagger_eng")  # POS tagger (English-specific version)
nltk.download("averaged_perceptron_tagger")  # General POS tagging model
nltk.download("wordnet")  # WordNet corpus for lemmatization and word relationships


# Load the required SpaCy model
nlp = spacy.load("en_core_web_sm")  # Load small English NLP model for tokenization, POS tagging, and dependency parsing

print("All dependencies loaded successfully!")  # Confirmation message

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


All dependencies loaded successfully!


# Final Inference Model

In [10]:
# Load spaCy model for dependency parsing
nlp = spacy.load('en_core_web_sm')

# Load the GloVe model
print("Loading GloVe model. This may take some time...")
glove_model = api.load("glove-wiki-gigaword-100")

# Load the trained models and vectorizer
print("Loading trained models and vectorizer...")
svm_model = joblib.load("svm_model.pkl")  # Load the trained SVM model
xgb_model = joblib.load("xgb_model.pkl")  # Load the trained XGBoost model
vectorizer = joblib.load("tfidf_vectorizer.pkl")  # Load the saved TF-IDF vectorizer

# Mapping of numeric labels to relation names
relation_mapping = {
    '0': 'Cause-Effect(e1,e2)', '1': 'Cause-Effect(e2,e1)',
    '2': 'Component-Whole(e1,e2)', '3': 'Component-Whole(e2,e1)',
    '4': 'Content-Container(e1,e2)', '5': 'Content-Container(e2,e1)',
    '6': 'Entity-Destination(e1,e2)', '7': 'Entity-Destination(e2,e1)',
    '8': 'Entity-Origin(e1,e2)', '9': 'Entity-Origin(e2,e1)',
    '10': 'Instrument-Agency(e1,e2)', '11': 'Instrument-Agency(e2,e1)',
    '12': 'Member-Collection(e1,e2)', '13': 'Member-Collection(e2,e1)',
    '14': 'Message-Topic(e1,e2)', '15': 'Message-Topic(e2,e1)',
    '16': 'Product-Producer(e1,e2)', '17': 'Product-Producer(e2,e1)',
    '18': 'Other'  # Default label for no clear relationship
}

def get_dependency_features(sentence):
    doc = nlp(sentence)  # Parse the sentence using spaCy
    features = []

    # Loop through the tokens (words) in the sentence
    for token in doc:
        # Feature: word, its POS tag, its dependency relation, and the head word of the token
        features.append(f"{token.text}_{token.pos}_{token.dep}_{token.head.text}")

    # Return a string of all the features
    return " ".join(features)

def get_glove_vector(sentence):
    """
    This function generates a GloVe vector for a given sentence.
    It tokenizes the sentence, looks up the corresponding GloVe vectors for the words,
    and returns the average of the vectors. If a word does not exist in the GloVe model,
    it is ignored.
    """
    words = word_tokenize(sentence)  # Tokenize the sentence into words
    vectors = [glove_model[word] for word in words if word in glove_model]  # Get vectors for words in the sentence
    return sum(vectors) / len(vectors) if vectors else np.zeros(100)  # Return the average vector or a zero vector if no valid words

def predict_relation(sentence, model):
    # Extract syntactic features and generate GloVe vector
    dependency_features = get_dependency_features(sentence)
    glove_vector = get_glove_vector(sentence)

    # Vectorize the input sentence using the loaded TF-IDF vectorizer
    X_input = vectorizer.transform([dependency_features])

    # Combine TF-IDF and GloVe vectors
    X_input_combined = hstack((X_input, glove_vector.reshape(1, -1)))

    # Predict the relation using the selected model
    prediction = model.predict(X_input_combined)

    # Map numeric label to relation name
    return relation_mapping.get(str(prediction[0]), "Unknown Relation")

if __name__ == "__main__":
    print("Welcome to the Relation Extraction System!")
    print("Enter a sentence annotated with <e1> and <e2> tags to predict the relation between entities.")

    while True:
        sentence = input("\nEnter your sentence (or 'exit' to quit): ").strip()

        if sentence.lower() == 'exit':
            print("Exiting the system. Thank you for using the Relation Extraction tool.")
            break

        if len(sentence) < 5:
            print("Please enter a valid sentence with at least 5 characters.")
            continue

        # Allow user to choose between SVM and XGBoost models for each prediction
        model_choice_svm = input("Predict using 'svm' model? (yes/no): ").strip().lower()
        model_choice_xgb = input("Predict using 'xgb' model? (yes/no): ").strip().lower()

        # Predict using SVM model if the user chooses 'yes'
        if model_choice_svm == 'yes':
            svm_relation = predict_relation(sentence, svm_model)
            print(f"Predicted Relation using SVM: {svm_relation}")

        # Predict using XGBoost model if the user chooses 'yes'
        if model_choice_xgb == 'yes':
            xgb_relation = predict_relation(sentence, xgb_model)
            print(f"Predicted Relation using XGBoost: {xgb_relation}")

        # If neither model is chosen
        if model_choice_svm != 'yes' and model_choice_xgb != 'yes':
            print("You must choose at least one model to predict.")

Loading GloVe model. This may take some time...
Loading trained models and vectorizer...
Welcome to the Relation Extraction System!
Enter a sentence annotated with <e1> and <e2> tags to predict the relation between entities.

Enter your sentence (or 'exit' to quit): The <e1>student</e1> <e2>association</e2> is the voice of the undergraduate student population of the State University of New York at Buffalo.
Predict using 'svm' model? (yes/no): yes
Predict using 'xgb' model? (yes/no): yes
Predicted Relation using SVM: Member-Collection(e1,e2)
Predicted Relation using XGBoost: Member-Collection(e1,e2)

Enter your sentence (or 'exit' to quit): Texas-born <e1>virtuoso</e1> finds harmony, sophistication in Appalachian <e2>instrument</e2>.
Predict using 'svm' model? (yes/no): yes
Predict using 'xgb' model? (yes/no): yes
Predicted Relation using SVM: Instrument-Agency(e2,e1)
Predicted Relation using XGBoost: Instrument-Agency(e2,e1)

Enter your sentence (or 'exit' to quit): exit
Exiting the sy